In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from exopt import DATA

OUTPUT_PATH = DATA / "output"

# I - Data Treatment

In [ ]:
filename = "20260504-114700-results_log.csv"

In [ ]:
data = pd.read_csv(OUTPUT_PATH/filename, index_col=0)

In [ ]:
data["culling_threshold"] = data["culling_threshold"].fillna(0)
data["culling_threshold"] = data["culling_threshold"].astype(float)
data["culling_threshold_label"] = data["culling_threshold"].apply(lambda x: f"Culling = {x:.2E}")

# Define ref values as matrix iterative results
ref_values = data[data["computation_type"]=="matrix_iterative"]

# Merge data and ref values to compute score precision
data = pd.merge(data, ref_values[["activity", "method", "score"]].rename(columns={"score":"ref_score"}), how="left", on=["activity", "method"])
data["score_precision"] = data[["score", "ref_score"]].apply(lambda row: 1-(row["ref_score"]-row["score"])/row["ref_score"], axis=1)

# Compute computation time efficiency as inverse of computation time for better graph representation
data["computation_time_eff"] = data["computation_time"].apply(lambda x: 1/x)

In [ ]:
data_for_scatter_plot = data[["computation_type", "culling_threshold", "score_precision", "computation_time"]].groupby(["computation_type", "culling_threshold"]).mean().reset_index()

# II. Plots

In [ ]:
fig, (ax1) = plt.subplots(figsize=(14, 6))
scatter = sns.scatterplot(
    data=data_for_scatter_plot.loc[data_for_scatter_plot["computation_type"]!="matrix_iterative"],
    x="computation_time",
    y="score_precision",
    hue="computation_type",
    style="culling_threshold",
    palette="deep",
    s=100,
    alpha=1,
    ax=ax1,
)

ax1.set_title(
    "Score Precision vs Computation Time per LCA computation (Average results for 100 datapoints per serie)"
)
ax1.set_xlabel("Computation Time [s]")
ax1.set_ylabel("Score Precision [%]")
ax1.legend(bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.show()

In [ ]:
NB_CUL_TREHS = 7 # used to offset the conditional means and align with stripplot

# Initialize the figure
f, ax1 = plt.subplots(figsize=(14, 6))
sns.despine(bottom=True, left=True)

# Show each observation with a scatterplot
sns.stripplot(
    data=data, x="score_precision", y="computation_type", hue="culling_threshold_label",
    dodge=True, jitter=False, alpha=0.3, zorder=1, legend=False, palette="deep", ax=ax1
)

# Show the conditional means, aligning each pointplot in the
# center of the strips by adjusting the width allotted to each
# category (.8 by default) by the number of hue levels
sns.pointplot(
    data=data,  x="score_precision", y="computation_type", hue="culling_threshold_label", palette="deep", errorbar=None, dodge=0.8-0.8/NB_CUL_TREHS,
    markers="d", markersize=6, linestyle="none",ax=ax1,
)

ax1.set_title("Computation type vs score precision per LCA computation", weight="bold")
ax1.set_xlabel("score precision [%]")
ax1.set_ylabel("culling threshold")

ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax1.text(x=0.25, y=-0.4, s="Opac diamond points represent conditional means while transparent dots represent individual observations", fontsize=10)
# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
# Initialize the figure
f, ax1 = plt.subplots(figsize=(14, 6))
sns.despine(bottom=True, left=True)
# Show each observation with a scatterplot
sns.stripplot(
    data=data, x="computation_time", y="computation_type", hue="culling_threshold_label",
    dodge=True, jitter=False, alpha=0.4, zorder=1, legend=False, palette="deep",ax=ax1
)

# Show the conditional means, aligning each pointplot in the
# center of the strips by adjusting the width allotted to each
# category (.8 by default) by the number of hue levels
sns.pointplot(
    data=data,  x="computation_time", y="computation_type", hue="culling_threshold_label", palette="deep", errorbar=None, dodge=0.8-0.8/NB_CUL_TREHS,
    markers="d", markersize=6, linestyle="none",ax=ax1
)

ax1.set_title("Computation type vs computation time per LCA computation")
ax1.set_xlabel("computation time [s]")
ax1.set_ylabel("culling threshold")

ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
total_comp_time_data = data[["computation_type", "culling_threshold_label", "score_precision", "computation_time"]].groupby(["computation_type", "culling_threshold_label"], sort=False).agg({"computation_time":"sum", "score_precision":"mean"}).reset_index()

In [ ]:
# Initialize the figure
f, ax1 = plt.subplots(figsize=(14, 6))
sns.despine()

# Show the conditional means, aligning each pointplot in the
# center of the strips by adjusting the width allotted to each
# category (.8 by default) by the number of hue levels
sns.pointplot(
    data=data,  x="computation_time", y="computation_type", hue="culling_threshold_label", palette="deep", errorbar=None, dodge=0.8-0.8/7,
    markers="d", markersize=6, linestyle="none",ax=ax1, estimator="sum"
)

ax1.set_title("Total computation time per computation type for 100 LCA computations")
ax1.set_xlabel("computation time [s] (log scale)")
ax1.set_ylabel("culling threshold")
ax1.set_xscale("log")
ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
data_for_cumulated_chart = data[["computation_type", "computation_time", "culling_threshold"]]
data_for_cumulated_chart = data_for_cumulated_chart[(data_for_cumulated_chart["culling_threshold"]==1e-5) | (data_for_cumulated_chart["culling_threshold"]==0)]
data_for_cumulated_chart["cumulative_computation_time"] = data_for_cumulated_chart.groupby("computation_type")["computation_time"].cumsum()
data_for_cumulated_chart["iteration"] = data_for_cumulated_chart.groupby("computation_type")["computation_type"].cumcount()+1

In [ ]:
# Initialize the figure
f, ax1 = plt.subplots(figsize=(14, 6))
sns.despine(bottom=True, left=True)

# Show each observation with a scatterplot
sns.lineplot(
    data=data_for_cumulated_chart, x="iteration", y="cumulative_computation_time", hue="computation_type",ax=ax1
)

ax1.set_title("Cumulative computation time per computation type")
ax1.set_xlabel("Iteration number")
ax1.set_ylabel("Cumulative computation time (s)")

ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')


# Show the plot
plt.tight_layout()
plt.show()